In [0]:
%sql
DROP VOLUME IF EXISTS uber.bronze.my_volume;
CREATE VOLUME uber.bronze.my_volume;


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *

# Event Hubs configuration
EH_NAMESPACE                    = "Rideuberevents"
EH_NAME                         = "rideubertopic"

#EH_CONN_STR                     = spark.conf.get("connection_string")
EH_CONN_STR                     = "Endpoint=sb://fake-eventhub-namespace.servicebus.windows.net/;SharedAccessKeyName=fake;SharedAccessKey=YOUR_SECRET_KEY_HERE"

# Kafka Consumer configuration
KAFKA_OPTIONS = {
  "kafka.bootstrap.servers"  : f"{EH_NAMESPACE}.servicebus.windows.net:9093",
  "subscribe"                : EH_NAME,
  "kafka.sasl.mechanism"     : "PLAIN",
  "kafka.security.protocol"  : "SASL_SSL",
  "kafka.sasl.jaas.config"   : f"kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"$ConnectionString\" password=\"{EH_CONN_STR}\";",
  "kafka.request.timeout.ms" : 10000,
  "kafka.session.timeout.ms" : 10000,
  "maxOffsetsPerTrigger"     : 10000,
  "failOnDataLoss"           : 'true',
  "startingOffsets"          : 'earliest'
}

df= spark.readStream.format("kafka")\
        .options(**KAFKA_OPTIONS)\
        .load()

display(df,checkpointLocation ="/Volumes/uber/bronze/my_volume")

In [0]:
batch_df = spark.read.format("kafka")\
                .options(**KAFKA_OPTIONS)\
                .load()
display(batch_df.limit(10))

In [0]:
%sql
DESCRIBE EXTENDED uber.bronze.bulk_rides

In [0]:

df=spark.read.table('uber.bronze.bulk_rides')
df.printSchema()

In [0]:
ddl_schema_string = df.schema.toDDL()
print(ddl_schema_string)

In [0]:
%sql
SELECT * FROM uber.bronze.bulk_stream_rides

In [0]:
%sql
SELECT * FROM uber.bronze.silver_obt